# Map SAML-D Dataset to Demodata Format

This notebook converts the SAML-D.csv dataset into the three demodata files
expected by the AML pipeline:

- **party.csv** — `partyId, partyType`
- **transactions.csv** — `tran_id, tx_type, base_amt, tran_timestamp, src, dst`
- **alert_transactions.csv** — `alert_id, alert_type, is_sar, tran_id`

### Mapping Rules
1. `partyType` = `Individual` for all parties
2. `partyId` = 8-char hex hash of each unique account number
3. `tran_timestamp` = `Date` + `Time` → ISO 8601 format
4. `tran_id` = auto-generated sequential integer
5. Extra columns (currencies, locations) preserved in a separate reference file
6. `party.csv` built from unique sender + receiver account IDs

## 1. Load SAML-D.csv

In [ ]:
import pandas as pd
import hashlib
import os

SAML_D_PATH = "/mnt/e/xx/SAML-D.csv"
OUTPUT_DIR = "/mnt/e/xx/demodata"

print(f"Output directory: {OUTPUT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(SAML_D_PATH)
print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Quick data overview
print("=== Is_laundering distribution ===")
print(df["Is_laundering"].value_counts())
print(f"\nLaundering rate: {df['Is_laundering'].mean():.4%}")
print(f"\n=== Laundering_type values ===")
print(df["Laundering_type"].value_counts())
print(f"\n=== Payment_type values ===")
print(df["Payment_type"].value_counts())
print(f"\nUnique senders: {df['Sender_account'].nunique():,}")
print(f"Unique receivers: {df['Receiver_account'].nunique():,}")

## 2. Generate partyId (8-char hex hash) for each account

In [ ]:
def account_to_party_id(account_number):
    """Convert an account number to an 8-char hex hash matching demodata format."""
    return hashlib.md5(str(account_number).encode()).hexdigest()[:8]

# Collect all unique accounts from both sender and receiver columns
all_accounts = pd.concat([
    df["Sender_account"],
    df["Receiver_account"]
]).unique()

print(f"Total unique accounts: {len(all_accounts):,}")

# Build account -> partyId lookup
account_to_pid = {acct: account_to_party_id(acct) for acct in all_accounts}

# Check for hash collisions
unique_hashes = len(set(account_to_pid.values()))
print(f"Unique hashes: {unique_hashes:,}")
if unique_hashes < len(all_accounts):
    print(f"WARNING: {len(all_accounts) - unique_hashes} hash collisions detected!")
else:
    print("No hash collisions.")

# Map to dataframe
df["src"] = df["Sender_account"].map(account_to_pid)
df["dst"] = df["Receiver_account"].map(account_to_pid)

print(f"\nSample mappings:")
print(df[["Sender_account", "src", "Receiver_account", "dst"]].head())

## 3. Create tran_timestamp (ISO 8601)

In [ ]:
# Combine Date + Time into ISO 8601 timestamp matching demodata format
# Demodata format: "2020-01-01T00:00:00.000Z"
df["tran_timestamp"] = pd.to_datetime(
    df["Date"] + "T" + df["Time"]
).dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")

print("Sample timestamps:")
print(df[["Date", "Time", "tran_timestamp"]].head())

## 4. Auto-generate tran_id

In [ ]:
# Sequential transaction ID starting from 1
df["tran_id"] = range(1, len(df) + 1)

print(f"tran_id range: {df['tran_id'].min()} to {df['tran_id'].max()}")

## 5. Map Payment_type → tx_type

Demodata uses: `TRANSFER-FanOut`, `TRANSFER-FanIn`, `TRANSFER-Forward`, `TRANSFER-Mutual`, `TRANSFER-Periodical`

SAML-D uses: `ACH`, `Cash Deposit`, `Cash Withdrawal`, `Cheque`, `Credit card`, `Cross-border`, `Debit card`

We map SAML-D payment types to the closest demodata equivalents while preserving the original value.

In [ ]:
# Map SAML-D Payment_type to pipeline-compatible tx_type
# The pipeline encodes tx_type to numeric codes in notebook 1:
#   CASH_IN=0, CASH_OUT=1, DEBIT=2, PAYMENT=3, TRANSFER=4, unknown=99
#
# We map to the broader categories the pipeline already handles:
PAYMENT_TYPE_MAP = {
    "Cash Deposit":   "CASH_IN",
    "Cash Withdrawal": "CASH_OUT",
    "ACH":            "TRANSFER-ACH",
    "Cheque":         "PAYMENT-Cheque",
    "Credit card":    "PAYMENT-CreditCard",
    "Debit card":     "DEBIT-Card",
    "Cross-border":   "TRANSFER-CrossBorder",
}

df["tx_type"] = df["Payment_type"].map(PAYMENT_TYPE_MAP)

# Verify no unmapped values
unmapped = df["tx_type"].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} rows have unmapped Payment_type:")
    print(df.loc[df["tx_type"].isna(), "Payment_type"].unique())
else:
    print("All Payment_type values mapped successfully.")

print("\ntx_type distribution:")
print(df["tx_type"].value_counts())

## 6. Map Laundering_type → alert_type

Demodata alert types: `gather_scatter`, `scatter_gather`, `cycle`

SAML-D laundering types are richer — we group them into the demodata categories
where possible and create new categories for the rest.

In [ ]:
# Map SAML-D Laundering_type to alert_type
# Direct matches + logical groupings
LAUNDERING_TYPE_MAP = {
    # --- Direct matches ---
    "Cycle":              "cycle",
    "Gather-Scatter":     "gather_scatter",
    "Scatter-Gather":     "scatter_gather",
    # --- Fan patterns ---
    "Fan_In":             "fan_in",
    "Fan_Out":            "fan_out",
    "Layered_Fan_In":     "fan_in",
    "Layered_Fan_Out":    "fan_out",
    # --- Structuring / smurfing ---
    "Smurfing":           "structuring",
    "Structuring":        "structuring",
    # --- Bipartite patterns ---
    "Bipartite":          "bipartite",
    "Stacked Bipartite":  "bipartite",
    # --- Other laundering ---
    "Deposit-Send":       "deposit_send",
    "Cash_Withdrawal":    "cash_withdrawal",
    "Over-Invoicing":     "over_invoicing",
    "Single_large":       "single_large",
    "Behavioural_Change_1": "behavioural_change",
    "Behavioural_Change_2": "behavioural_change",
}

# Only laundering rows get an alert_type
df["alert_type"] = df["Laundering_type"].map(LAUNDERING_TYPE_MAP)

# Normal rows should have no alert_type
df.loc[df["Is_laundering"] == 0, "alert_type"] = None

print("=== alert_type distribution (laundering rows only) ===")
print(df.loc[df["Is_laundering"] == 1, "alert_type"].value_counts())

# Check for unmapped laundering rows
unmapped_launder = df[(df["Is_laundering"] == 1) & (df["alert_type"].isna())]
if len(unmapped_launder) > 0:
    print(f"\nWARNING: {len(unmapped_launder)} laundering rows unmapped:")
    print(unmapped_launder["Laundering_type"].unique())
else:
    print("\nAll laundering types mapped successfully.")

## 7. Build party.csv

Collect unique partyIds from both `src` and `dst`. All set to `Individual`.

In [ ]:
# Unique party IDs from both sender and receiver
party_ids = sorted(set(df["src"].unique()) | set(df["dst"].unique()))

party_df = pd.DataFrame({
    "partyId": party_ids,
    "partyType": "Individual"
})

print(f"party.csv: {len(party_df):,} unique parties")
party_df.head()

## 8. Build transactions.csv

Columns: `tran_id, tx_type, base_amt, tran_timestamp, src, dst`

In [ ]:
transactions_df = df[["tran_id", "tx_type", "Amount", "tran_timestamp", "src", "dst"]].copy()
transactions_df = transactions_df.rename(columns={"Amount": "base_amt"})

print(f"transactions.csv: {len(transactions_df):,} rows")
print(f"\nColumn dtypes:")
print(transactions_df.dtypes)
transactions_df.head()

## 9. Build alert_transactions.csv

Columns: `alert_id, alert_type, is_sar, tran_id`

Only includes rows where `Is_laundering == 1`. Each unique `alert_type` group
gets an `alert_id`.

In [ ]:
# Filter to laundering rows only
launder_df = df[df["Is_laundering"] == 1].copy()

# Assign alert_id: group by alert_type so each type gets sequential IDs
# In demodata, alert_id groups related transactions of the same alert
# We assign one alert_id per unique (alert_type) group, incrementing sequentially
alert_type_to_id = {atype: idx + 1 for idx, atype in enumerate(sorted(launder_df["alert_type"].unique()))}
launder_df["alert_id"] = launder_df["alert_type"].map(alert_type_to_id)

alert_transactions_df = launder_df[["alert_id", "alert_type", "tran_id"]].copy()
alert_transactions_df["is_sar"] = "true"

# Reorder columns to match demodata format
alert_transactions_df = alert_transactions_df[["alert_id", "alert_type", "is_sar", "tran_id"]]

print(f"alert_transactions.csv: {len(alert_transactions_df):,} rows")
print(f"\nalert_id mapping:")
for atype, aid in sorted(alert_type_to_id.items(), key=lambda x: x[1]):
    count = (alert_transactions_df["alert_type"] == atype).sum()
    print(f"  {aid}: {atype} ({count:,} transactions)")

alert_transactions_df.head()

## 10. Save extra information

Preserve the SAML-D columns not in demodata format as a reference file.

In [ ]:
# Extra columns: tran_id + original SAML-D fields not in demodata
extra_df = df[[
    "tran_id",
    "Sender_account", "Receiver_account",
    "Payment_currency", "Received_currency",
    "Sender_bank_location", "Receiver_bank_location",
    "Payment_type", "Laundering_type", "Is_laundering"
]].copy()

print(f"extra_info.csv: {len(extra_df):,} rows, {len(extra_df.columns)} columns")
extra_df.head()

## 11. Save all output files

In [ ]:
# Save to demodata directory
party_path = os.path.join(OUTPUT_DIR, "party.csv")
transactions_path = os.path.join(OUTPUT_DIR, "transactions.csv")
alerts_path = os.path.join(OUTPUT_DIR, "alert_transactions.csv")
extra_path = os.path.join(OUTPUT_DIR, "extra_info.csv")

party_df.to_csv(party_path, index=False)
transactions_df.to_csv(transactions_path, index=False)
alert_transactions_df.to_csv(alerts_path, index=False)
extra_df.to_csv(extra_path, index=False)

print("=== Files saved ===")
for name, path in [("party.csv", party_path),
                    ("transactions.csv", transactions_path),
                    ("alert_transactions.csv", alerts_path),
                    ("extra_info.csv", extra_path)]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {name}: {size_mb:.1f} MB -> {path}")

## 12. Validation — compare output to demodata format

In [ ]:
# Validate output matches demodata schema
print("=== party.csv ===")
p = pd.read_csv(party_path)
print(f"  Columns: {list(p.columns)}")
print(f"  Rows: {len(p):,}")
print(f"  partyType unique: {p['partyType'].unique()}")
print(f"  partyId sample: {p['partyId'].head(3).tolist()}")
print(f"  partyId length: {p['partyId'].str.len().unique()}")

print("\n=== transactions.csv ===")
t = pd.read_csv(transactions_path)
print(f"  Columns: {list(t.columns)}")
print(f"  Rows: {len(t):,}")
print(f"  tx_type unique: {sorted(t['tx_type'].unique())}")
print(f"  timestamp sample: {t['tran_timestamp'].head(3).tolist()}")

print("\n=== alert_transactions.csv ===")
a = pd.read_csv(alerts_path)
print(f"  Columns: {list(a.columns)}")
print(f"  Rows: {len(a):,}")
print(f"  alert_type unique: {sorted(a['alert_type'].unique())}")
print(f"  is_sar unique: {a['is_sar'].unique()}")

# Cross-check: all alert tran_ids exist in transactions
missing = set(a["tran_id"]) - set(t["tran_id"])
print(f"  Alert tran_ids missing from transactions: {len(missing)}")

# Cross-check: all src/dst exist in party
all_party_ids = set(p["partyId"])
missing_src = set(t["src"]) - all_party_ids
missing_dst = set(t["dst"]) - all_party_ids
print(f"\n  Src partyIds missing from party.csv: {len(missing_src)}")
print(f"  Dst partyIds missing from party.csv: {len(missing_dst)}")

print("\n=== Validation complete ===")